Rosa Kalista Rahma Dwi Pramesti

23.11.5662

23 IF 06

**UAS Big Data & Predictive Analytics Lanjut**

Dosen Pengampu : Mulia Sulistiyono, M.Kom

Tujuan dari kode PySpark yang dibuat adalah untuk memprediksi Total Amount (Total Penjualan) pada transaksi retail.

Perkiraan nilai total penjualan

Bisa digunakan untuk:
- Estimasi pendapatan
- Analisis performa penjualan
- Perencanaan stok

### **Inisialisasi Spark session**

In [13]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviewsBigData") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

### **Load Dataset**

In [14]:
df = spark.read.csv(
    "/content/drive/MyDrive/smt 5/BIG DATA LANJUT/UAS/retail_sales_dataset.csv",
    header=True,
    inferSchema=True
)

In [15]:
df.printSchema()
df.show(5)

root
 |-- Transaction ID: integer (nullable = true)
 |-- Date: date (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Price per Unit: integer (nullable = true)
 |-- Total Amount: integer (nullable = true)

+--------------+----------+-----------+------+---+----------------+--------+--------------+------------+
|Transaction ID|      Date|Customer ID|Gender|Age|Product Category|Quantity|Price per Unit|Total Amount|
+--------------+----------+-----------+------+---+----------------+--------+--------------+------------+
|             1|2023-11-24|    CUST001|  Male| 34|          Beauty|       3|            50|         150|
|             2|2023-02-27|    CUST002|Female| 26|        Clothing|       2|           500|        1000|
|             3|2023-01-13|    CUST003|  Male| 50|     Electronics|       1|            30

## **EDA**

#### **Jumlah Data**

In [16]:
print("Jumlah baris:", df.count())
print("Jumlah kolom:", len(df.columns))

Jumlah baris: 1000
Jumlah kolom: 9


EDA diawali dengan memahami skala data, sehingga dapat ditentukan metode preprocessing dan model yang sesuai.

#### **Statistik deskriptif**

In [17]:
df.describe().show()

+-------+-----------------+-----------+------+------------------+----------------+------------------+------------------+-----------------+
|summary|   Transaction ID|Customer ID|Gender|               Age|Product Category|          Quantity|    Price per Unit|     Total Amount|
+-------+-----------------+-----------+------+------------------+----------------+------------------+------------------+-----------------+
|  count|             1000|       1000|  1000|              1000|            1000|              1000|              1000|             1000|
|   mean|            500.5|       NULL|  NULL|            41.392|            NULL|             2.514|            179.89|            456.0|
| stddev|288.8194360957494|       NULL|  NULL|13.681429659122518|            NULL|1.1327343409145354|189.68135627129234|559.9976315551235|
|    min|                1|    CUST001|Female|                18|          Beauty|                 1|                25|               25|
|    max|             1000|

Menampilkan statistik dasar seperti:
- Mean
- Min
- Max
- Standar deviasi

Statistik ini membantu:
- Mengidentifikasi outlier
- Menilai sebaran data
- Menentukan apakah data perlu normalisasi

#### **Total penjualan per kategori**

In [18]:
df.groupBy("Product Category") \
  .sum("Total Amount") \
  .withColumnRenamed("sum(Total Amount)", "TotalSales") \
  .orderBy("TotalSales", ascending=False) \
  .show()

+----------------+----------+
|Product Category|TotalSales|
+----------------+----------+
|     Electronics|    156905|
|        Clothing|    155580|
|          Beauty|    143515|
+----------------+----------+



Menghitung total penjualan berdasarkan kategori produk. Analisis ini memberikan value bisnis, yaitu mengetahui kategori dengan kontribusi penjualan terbesar.

### **PREPROCESSING DATA**

#### **Memilih kolom yang penting**

In [20]:
df_clean = df.select(
    "Product Category",
    "Price per Unit",
    "Quantity",
    "Total Amount"
)

Memilih kolom yang relevan terhadap tujuan prediksi. Fungsinya tuntuk mengurangi kolom yang tidak relevan sehingga mempercepat komputasi dan mengurangi noise pada model.

#### **Handling Missing Value**

In [21]:
df_clean = df_clean.dropna()

Menghapus baris data yang memiliki nilai kosong. Karena data kosong dapat menyebabkan error pada training model dan prediksi tidak akurat, sehingga data harus dibersihkan sebelum masuk ke tahap ML.

#### **Casting tipe data**

In [22]:
from pyspark.sql.functions import col

df_clean = df_clean \
    .withColumn("Price per Unit", col("Price per Unit").cast("double")) \
    .withColumn("Quantity", col("Quantity").cast("int")) \
    .withColumn("Total Amount", col("Total Amount").cast("double"))

Menyesuaikan tipe data agar sesuai dengan kebutuhan komputasi numerik. ML hanya menerima data numerik, sehingga casting tipe data adalah tahap wajib dalam mempreprocessing.

### **SPARK SQL**

In [23]:
df_clean.createOrReplaceTempView("retail")

####**Total & rata-rata penjualan per kategori**

In [25]:
spark.sql("""
    SELECT
        `Product Category`,
        COUNT(*) AS total_transaksi,
        AVG(`Total Amount`) AS avg_penjualan,
        SUM(`Total Amount`) AS total_penjualan
    FROM retail
    GROUP BY `Product Category`
    ORDER BY total_penjualan DESC
""").show()

+----------------+---------------+-----------------+---------------+
|Product Category|total_transaksi|    avg_penjualan|total_penjualan|
+----------------+---------------+-----------------+---------------+
|     Electronics|            342|458.7865497076023|       156905.0|
|        Clothing|            351|443.2478632478632|       155580.0|
|          Beauty|            307|467.4755700325733|       143515.0|
+----------------+---------------+-----------------+---------------+



Untuk menghitung jumlah transaksi, rata-rata penjualan, dan total penjualan perkategori. Fungsinya untuk memudahkan analisis data besar menggunakan bahasa SQL yang familiar, sekaligus memanfaatkan optimasi Spark

### **PEMROSESAN BATCH (MAPREDUCE DENGAN RDD)**

In [26]:
rdd = df_clean.rdd

####**Map → (Category, Total Amount)**

In [29]:
pair_rdd = rdd.map(lambda x: (x["Product Category"], x["Total Amount"]))

####**ReduceByKey (total penjualan per kategori)**

In [31]:
total_sales = pair_rdd.reduceByKey(lambda a, b: a + b)
total_sales.collect()

[('Beauty', 143515.0), ('Clothing', 155580.0), ('Electronics', 156905.0)]

Implementasi konsep MapReduce:
- Map: membuat pasangan key-value
- Reduce: agregasi nilai

### **OPERASI RDD**

**Map**

In [30]:
mapped = rdd.map(lambda x: (x["Product Category"], 1))

**GroupByKey**

In [32]:
grouped = mapped.groupByKey().mapValues(len)
grouped.collect()

[('Beauty', 307), ('Clothing', 351), ('Electronics', 342)]

Mengelompokkan data berdasarkan kategori lalu menghitung jumlah elemen. Digunakan untuk analisis frekuensi transaksi per kategori.

**CombineByKey**

In [34]:
combined = rdd.map(lambda x: (x["Product Category"], x["Total Amount"])) \
    .combineByKey(
        lambda v: (v, 1),
        lambda acc, v: (acc[0] + v, acc[1] + 1),
        lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
    )

avg_sales = combined.mapValues(lambda x: x[0] / x[1])
avg_sales.collect()

[('Beauty', 467.4755700325733),
 ('Clothing', 443.2478632478632),
 ('Electronics', 458.7865497076023)]

Menghitung rata-rata penjualan per kategori secara efisien. CombineByKey lebih optimal dibanding GroupByKey karena:
- Mengurangi data shuffle
- Lebih efisien untuk Big Data

### **FEATURE ENGGINEERING (ML PIPELINE)**

In [35]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

**Encode kategori**

In [40]:
indexer = StringIndexer(
    inputCol="Product Category",
    outputCol="CategoryIndex"
)

Mengubah data kategori (string) menjadi angka. Model ML tidak bisa memproses data kategorikal langsung → perlu encoding.

**Gabungkan fitur**

In [37]:
assembler = VectorAssembler(
    inputCols=["CategoryIndex", "Price per Unit", "Quantity"],
    outputCol="features"
)

Menggabungkan semua fitur menjadi satu vektor. MLlib membutuhkan satu kolom fitur berbentuk vektor agar model dapat dilatih.

### **SPLIT DATA**

In [38]:
train_data, test_data = df_clean.randomSplit([0.8, 0.2], seed=42)

Membagi data menjadi data latih dan data uji. Pemisahan ini penting untuk:
- Menghindari overfitting
- Mengukur performa model secara objektif

### **MODELING**

#### **Linear Regression**

In [41]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline

lr = LinearRegression(
    featuresCol="features",
    labelCol="Total Amount"
)

pipeline_lr = Pipeline(stages=[
    indexer,
    assembler,
    lr
])

model_lr = pipeline_lr.fit(train_data)

Digunakan untuk memodelkan hubungan linier antara fitur dan total penjualan.

#### **Decison Tree**

In [51]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="Total Amount"
)

pipeline_dt = Pipeline(stages=[
    indexer,
    assembler,
    dt
])

model_dt = pipeline_dt.fit(train_data)

Model non-linear berbasis pohon keputusan.

### **EVALUASI MODEL**

In [43]:
from pyspark.ml.evaluation import RegressionEvaluator

**Evaluator**

In [44]:
evaluator = RegressionEvaluator(
    labelCol="Total Amount",
    predictionCol="prediction",
    metricName="rmse"
)

**Linear Regression**

In [45]:
pred_lr = model_lr.transform(test_data)
rmse_lr = evaluator.evaluate(pred_lr)
print("RMSE Linear Regression:", rmse_lr)

RMSE Linear Regression: 218.6680906861407


**Decision Tree**

In [46]:
pred_dt = model_dt.transform(test_data)
rmse_dt = evaluator.evaluate(pred_dt)
print("RMSE Decision Tree:", rmse_dt)

RMSE Decision Tree: 0.0


RMSE mengukur rata-rata kesalahan prediksi. RMSE cocok untuk regresi karena:
- Sensitif terhadap error besar
- Mudah diinterpretasi dalam satuan asli data

### **HYPERPARAMETER TUNING**

In [47]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [48]:
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5]) \
    .build()

In [49]:
cv = CrossValidator(
    estimator=pipeline_lr,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3
)

cv_model = cv.fit(train_data)

Mencari kombinasi parameter terbaik secara otomatis.

### **EVALUASI MODEL**

In [50]:
best_pred = cv_model.transform(test_data)
best_rmse = evaluator.evaluate(best_pred)

print("Best Model RMSE:", best_rmse)

Best Model RMSE: 218.6693170270694
